# Getting started — SmartRing-Article

This notebook is the entry point for anyone (student, reviewer, future-you) who wants to see what the project contains, verify that the data bundle fits together, and learn the two or three patterns used everywhere in the codebase.

**What you will do here**

1. Set up the interactive DataFrame renderer (Colab) or the plain renderer (local Jupyter).
2. Preview the three contract files in `synthetic/v1/` — this is the interface WP6 builds against.
3. Preview a large raw mcPHASES file using column selection so it loads in under a second on Colab free tier.
4. Run the contract validator to confirm the bundle is internally consistent.

**Before running**

- If you are on Colab: mount your Drive copy of the repo, or clone it into the session. Then `%cd` into the repo root.
- If you are local: activate `.venv` and run `jupyter lab notebooks/00_getting_started.ipynb`.
- Either way, you need the synthetic bundle. Generate it with `python synthetic/generator.py --out synthetic/v1 --seed 42` if it is not present yet.
- For the raw-data cell you need your own PhysioNet access to mcPHASES (see `README.md`) and the output of `python scripts/convert_raw_to_parquet.py`.

## 1. Setup

Two lines enable Colab's interactive table renderer. If you are not on Colab the try/except falls through harmlessly.

In [ ]:
import sys, os
from pathlib import Path

# Make the repo root importable regardless of where the notebook lives.
REPO_ROOT = Path().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

import pandas as pd
from utils.preview import peek, summary

try:
    from google.colab import data_table
    data_table.enable_dataframe_formatter()
    print('Colab interactive table formatter: ON')
except ImportError:
    print('Local Jupyter: default DataFrame renderer')
print('Repo root:', REPO_ROOT)

## 2. The data contract — three parquet files in `synthetic/v1/`

See `docs/data_contract_v1.md` for the authoritative spec. In one sentence: WP5 emits `probability_table`, WP3 emits `labels`, WP2 emits `covariates`, and WP6 reads all three.

Preview each file. The output shows row count, column count, and per-column dtypes — then returns the first 10 rows as an interactive table.

In [ ]:
peek('synthetic/v1/probability_table.parquet')

In [ ]:
peek('synthetic/v1/labels.parquet')

In [ ]:
peek('synthetic/v1/covariates.parquet')

### Per-column summary

Use `summary()` for null counts, unique counts, and numeric min/max. This is what you reach for after `peek()` when something looks off.

In [ ]:
summary('synthetic/v1/probability_table.parquet')

## 3. Large raw files — column selection is how you survive Colab

Raw mcPHASES `heart_rate.csv` is 1.9 GB and blows up Colab's free-tier memory. After the parquet conversion (`scripts/convert_raw_to_parquet.py`) the same data is a 230 MB parquet file — and if you only need two columns, you can read *only those two columns* directly from disk.

**Pattern:** always pass `columns=[...]` to `peek` (or `pd.read_parquet`) when working with raw files on Colab.

In [ ]:
# If the converted file is not present, print a helpful note and skip.
hr_path = Path('dataset_parquet/heart_rate.parquet')
if hr_path.exists():
    peek(hr_path, columns=['id', 'day_in_study', 'bpm'], n=20)
else:
    print(f'{hr_path} not found.')
    print('Run `python scripts/convert_raw_to_parquet.py` after placing the raw mcPHASES CSVs in `dataset/`.')

## 4. Validate the synthetic bundle

`synthetic/validate_contract.py` checks every contract rule in `docs/data_contract_v1.md` — schema, dtypes, value ranges, referential integrity across the three files. Non-zero exit means the bundle is broken.

In [ ]:
import subprocess
result = subprocess.run(
    [sys.executable, 'synthetic/validate_contract.py', '--dir', 'synthetic/v1'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise SystemExit(f'Validator failed with exit {result.returncode}')

## Next steps

- **Student 3 (WP6):** see `docs/onboarding_student3.md`. Milestone 1 is reading. Milestone 2 is rebuilding / tuning the synthetic generator. Milestone 3 is the first conformal layer on synthetic data — the unblock gate.
- **Everyone:** read `docs/claims_boundary.md` and sign. Read `docs/student3_charter.md` to understand what Student 3's non-negotiable language rules are (they apply to everyone's drafts).
- **Working with new files:** `peek(path)` and `summary(path)` handle both parquet and CSV. When a file is large, pass `columns=[...]` to `peek` so only the columns you need load into memory.
- **When you change the contract:** the contract version in `docs/data_contract_v1.md` §9 must bump, the validator must update, and the synthetic bundle must be regenerated. Email the team before the change lands.